需求：

北京明天的天气怎么样？上海呢？ （通过两次对话实现）

以ReAct模式为例进行实现：



In [2]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI
from langchain_core.tools import StructuredTool, Tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor, AgentType, initialize_agent

import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
dotenv.load_dotenv()

# 读取配置文件的信息
# os.environ['TAVILY_API_KEY'] = os.getenv("TAVILY_API_KEY")
# os.environ['TAVILY_API_KEY'] = "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"
os.environ['TAVILY_API_KEY'] = "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"

# 获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)

# 获取一个搜索的工具
search_tool = Tool(
    func=search.run,
    name="Search",
    description="用于检索互联网上的信息",
)


# 获取大语言模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 获取一个记忆的实例：ConversationBufferMemory
# return_messages=True → 返回 对象列表（推荐，与 ChatPromptTemplate 配合）
#
# return_messages=False → 返回 纯文本字符串（与旧版 PromptTemplate 配合）
#
# 如果你使用的是现代 LangChain 写法（ChatPromptTemplate + MessagesPlaceholder），请务必设置 return_messages=True，并且 memory_key 的值要与 MessagesPlaceholder 的 variable_name 完全一致。


memory = ConversationBufferMemory(
    return_messages=True,
    memory_key="chat_history"  #此时必须设置memory_key,且值为chat_history。因为提示词模板中使用的是chat_history。二者必须一致！
)

# 获取Agent的实例
agent = AgentType.CONVERSATIONAL_REACT_DESCRIPTION

# 获取AgentExecutor的实例
# 源码中创建了默认的提示词模版，
'''
        template = f"{prefix}\n\n{tool_strings}\n\n{format_instructions}\n\n{suffix}"
        if input_variables is None:
            input_variables = ["input", "chat_history", "agent_scratchpad"]
        return PromptTemplate(template=template, input_variables=input_variables)
'''

agent_executor = initialize_agent(
    tools = [search_tool],
    llm = llm,
    agent = agent,
    verbose = True, # 显示详细的日志信息
    memory = memory,
)


# 通过AgentExecutor 调用invoke(),并得到响应
result = agent_executor.invoke("查询北京今天的天气情况")


# 处理响应数据
print(result)



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: Search
Action Input: 北京今天的天气情况
```
Observation: [{'title': '北京今日天气预报 - China Weather', 'url': 'https://chinaweather.org/zh-hans/beijing', 'content': '## 北京的天气\n\n 上海\n 四川\n 粤\n 天津\n 山西\n 重庆\n 甘肃\n 河北\n 陕西\n 新疆\n 福建\n 江苏\n 吉林\n 海南\n 云南\n 贵州\n 内蒙古\n 湖南\n 安徽\n 浙江\n 江西\n 湖北\n 黑龙江\n 河南\n 宁夏\n 辽宁\n 青海\n 山东\n 西藏\n 广西\n\n## 常见问题\n\n### 1. 北京 最热的月份是哪一个月？\n\n七月 的月平均气温最高（日平均值为 32.53°C），十二月 的月平均气温最低（日平均值为 -5.19°C）。\n\n### 2. 今天我应该在北京穿什么？\n\n今天北京天气温暖，气温在 19ºC 到 22ºC（66.2ºF 到 71.6ºF）之间，您可以考虑穿：  \n - 轻薄透气的袜子  \n - T恤或短袖衬衫  \n - 舒适的凉鞋或运动鞋  \n - 遮阳帽或鸭舌帽\n\n### 温度\n\nEarly\n\n-8°C / 17°C\n\n-5°C / 20°C\n\n-3°C / 14°C\n\n-2°C / 9°C\n\n### 空气质量\n\n缓和\n\n CO2 478\n CO 909\n NH3 0\n NO2 72\n O3 11\n PM10 70\n PM25 58\n SO2 42\n CH4 1994\n\n### 日出/日落\n\n### 北京 气象图 [...] 北京,  天气 \n\n(摄影： Henry Chen 在 Unsplash)\n\n \n\n## 目前情况\n\nClear Sky\n\n°C °F\n\n健康）状况: 晴朗的天空\n\n压力: 1009 hPa\n\nFeel like: 10 °C\n\n温度: 13°C\n\n云: 0%\n

In [3]:
result = agent_executor.invoke("上海呢？")
# 处理响应数据
print(result)
'''
{'input': '上海呢？', 'chat_history': [HumanMessage(content='查询北京今天的天气情况', additional_kwargs={}, response_metadata={}), AIMessage(content='今天北京的天气情况如下：\n\n- **天气**：多云\n- **气温**：最高约 18°C，最低约 7°C\n- **风速**：西南风 2 级\n- **湿度**：约 16%\n- **空气质量**：良好\n\n建议您穿着轻便舒适的衣物，适合外出活动。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='上海呢？', additional_kwargs={}, response_metadata={}), AIMessage(content='今天上海的天气情况如下：\n\n- **天气**：小雨\n- **气温**：白天最高约 16°C，夜间最低约 13°C\n- **风速**：北风 2 级\n- **湿度**：约 95%\n\n建议您出门时带上雨具，穿着轻便舒适的衣物。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'output': '今天上海的天气情况如下：\n\n- **天气**：小雨\n- **气温**：白天最高约 16°C，夜间最低约 13°C\n- **风速**：北风 2 级\n- **湿度**：约 95%\n\n建议您出门时带上雨具，穿着轻便舒适的衣物。'}
'''



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: Search
Action Input: 上海今天的天气情况
```
Observation: [{'title': '今天上海市天气_今日天气预报 - 墨迹天气', 'url': 'https://tianqi.moji.com/today/china/shanghai/shanghai', 'content': '17° 14°\n\n   中雨  中雨 04/08\n 周四 小雨  小雨 \n\n  20° 14°\n\n   小雨  小雨 04/09\n 周五 小雨  小雨 \n\n  18° 14°\n\n   小雨  小雨 04/10\n 周六 小雨  小雨 \n\n  21° 13°\n\n   小雨  小雨 04/11\n 周日 小雨  小雨 \n\n  19° 7°\n\n   阴  阴 04/12\n 周一 阴  阴 \n\n  16° 10°\n\n   阴  阴 04/13\n 周二 阴  阴 \n\n  16° 10°\n\n   阴  阴 04/14\n 周三 阴  阴 \n\n  18° 14°\n\n   阴  阴 04/15\n 周四 多云  多云 \n\n  22° 14°\n\n   阴  阴 04/16\n 周五 阴  阴 \n\n  20° 14°\n\n   多云  多云 04/17\n\n  \n\n  最弱\n  :   紫外线\n  \n\n  不易中暑\n  :   中暑\n  \n\n  易发\n  :   感冒\n  \n\n  较适宜\n  :   狗狗\n  \n\n  不适宜\n  :   猫咪\n  \n\n  不适宜\n  :   运动\n  \n\n  控油\n  :   化妆\n  \n\n  温凉\n  :   穿衣\n  \n\n  较适宜\n  :   广场舞\n  \n\n  较适宜\n  :   钓鱼\n  \n\n  一般\n  :   交通\n  \n\n  不适宜\n  :   划船\n  \n\n  较不适宜\n  :   啤酒\n  \n\n  较适宜\n  :   夜生活\n  \n\n  差\n 